# Tempest Weather Station — UDP Communication Test

This notebook listens for UDP broadcasts from a WeatherFlow Tempest hub on the local network.
The hub broadcasts JSON messages on **port 50222**. No cloud connection is required.

Run cells top-to-bottom on first use.

## 1. Imports & Configuration

In [3]:
import json
import socket
import time
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple

import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

UDP_PORT: int = 50222
BUFFER_SIZE: int = 4096

print(f"UDP port: {UDP_PORT}")
print("Ready.")

UDP port: 50222
Ready.


In [4]:
def get_sock(timeout=None):
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    if timeout:
        sock.settimeout(timeout)
    sock.bind(("", UDP_PORT))

    return sock

## 2. Single Message Listener

Opens a UDP socket and waits for one message. The hub sends a `hub_status` heartbeat every **10 seconds**, so you should see a message quickly.

In [5]:
def receive_one_message(timeout: int = 15) -> Tuple[dict, str]:
    """Wait for one UDP broadcast and return (parsed_json, source_ip)."""
    # sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    # sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    # sock.settimeout(timeout)
    # sock.bind(("", UDP_PORT))
    sock = get_sock(timeout)
    try:
        data, addr = sock.recvfrom(BUFFER_SIZE)
        return json.loads(data.decode("utf-8")), addr
    finally:
        sock.close()


print("Waiting for a message (up to 15 seconds)...")
msg, addr = receive_one_message()
source_ip = addr[0]
print(f"\nReceived from {source_ip} port:{addr[1]}")
print(json.dumps(msg, indent=2))

Waiting for a message (up to 15 seconds)...

Received from 192.168.68.57 port:57707
{
  "serial_number": "ST-00199400",
  "type": "rapid_wind",
  "hub_sn": "HB-00201189",
  "ob": [
    1779316668,
    0.0,
    0
  ]
}


## 3. Multi-Message Capture

Collects all UDP messages for a configurable duration and summarises what arrived.

In [6]:
def receive_messages(duration: int = 30) -> List[dict]:
    """Collect all UDP broadcasts for `duration` seconds."""
    messages: List[dict] = []
    # sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    # sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    # sock.settimeout(1.0)
    # sock.bind(("", UDP_PORT))
    sock = get_sock(timeout=1.0)
    deadline = time.monotonic() + duration
    try:
        while time.monotonic() < deadline:
            try:
                data, _ = sock.recvfrom(BUFFER_SIZE)
                messages.append(json.loads(data.decode("utf-8")))
            except socket.timeout:
                pass
    finally:
        sock.close()
    return messages


CAPTURE_SECONDS = 30  # adjust as needed
print(f"Capturing for {CAPTURE_SECONDS} seconds...")
captured = receive_messages(CAPTURE_SECONDS)

from collections import Counter
counts = Counter(m.get("type", "unknown") for m in captured)
print(f"\nCaptured {len(captured)} messages:")
for msg_type, count in sorted(counts.items()):
    print(f"  {msg_type:20s} {count}")

Capturing for 30 seconds...

Captured 13 messages:
  device_status        1
  hub_status           1
  obs_st               1
  rapid_wind           10


## 4. Tempest UDP Message Reference

| Type | Description | Typical Frequency |
|------|-------------|-------------------|
| `hub_status` | Hub heartbeat — uptime, RSSI, firmware | Every 10 s |
| `device_status` | Tempest sensor heartbeat — battery, sensor flags | Every 60 s |
| `obs_st` | Full weather observation array | Every 60 s (default) |
| `rapid_wind` | Wind speed + direction snapshot | Every 3 s |
| `evt_precip` | Rain start event | On event |
| `evt_strike` | Lightning strike — distance + energy | On event |

Full field reference: https://weatherflow.github.io/Tempest/api/udp/v171/

## 5. Parse `obs_st` — Weather Observations

`obs_st` packs all sensor readings into a positional array. The cell below labels them.

In [7]:
OBS_ST_FIELDS: List[str] = [
    "timestamp",
    "wind_lull_m_s",
    "wind_avg_m_s",
    "wind_gust_m_s",
    "wind_direction_deg",
    "wind_sample_interval_s",
    "pressure_mb",
    "air_temp_c",
    "relative_humidity_pct",
    "illuminance_lux",
    "uv_index",
    "solar_radiation_w_m2",
    "rain_prev_min_mm",
    "precip_type",
    "lightning_avg_dist_km",
    "lightning_count",
    "battery_volts",
    "report_interval_min",
]


def parse_obs_st(msg: dict) -> Optional[pd.DataFrame]:
    """Return obs_st observations as a labeled single-row DataFrame."""
    obs = msg.get("obs")
    if not obs:
        return None
    row = dict(zip(OBS_ST_FIELDS, obs[0]))
    row["timestamp"] = datetime.fromtimestamp(row["timestamp"], tz=timezone.utc)
    row["serial_number"] = msg.get("serial_number")
    return pd.DataFrame([row])


obs_messages = [m for m in captured if m.get("type") == "obs_st"]
if obs_messages:
    df = parse_obs_st(obs_messages[-1])
    display(df.T.rename(columns={0: "value"}))
else:
    print("No obs_st messages in capture. Wait ~60 s after the station reports.")

,value
timestamp,2026-05-20 22:38:19+00:00
wind_lull_m_s,0.0
wind_avg_m_s,0.02
wind_gust_m_s,0.22
wind_direction_deg,220
wind_sample_interval_s,3
pressure_mb,1009.99
air_temp_c,29.11
relative_humidity_pct,51.68
illuminance_lux,3797


## 6. Parse `rapid_wind` — Wind Snapshots

In [8]:
def parse_rapid_wind(msg: dict) -> Optional[Dict]:
    """Return timestamp, speed (m/s), direction (degrees) from a rapid_wind message."""
    ob = msg.get("ob")
    if not ob or len(ob) < 3:
        return None
    return {
        "timestamp": datetime.fromtimestamp(ob[0], tz=timezone.utc),
        "wind_speed_m_s": ob[1],
        "wind_speed_mph": ob[1] * 2.23694,
        "wind_direction_deg": ob[2],
        "serial_number": msg.get("serial_number"),
    }


wind_messages = [m for m in captured if m.get("type") == "rapid_wind"]
if wind_messages:
    wind_rows = [parse_rapid_wind(m) for m in wind_messages]
    df_wind = pd.DataFrame([r for r in wind_rows if r])
    display(df_wind.tail(10))
else:
    print("No rapid_wind messages in capture.")

,timestamp,wind_speed_m_s,wind_speed_mph,wind_direction_deg,serial_number
0,2026-05-20 22:37:54+00:00,0.00,0.000000,0,ST-00199400
1,2026-05-20 22:37:58+00:00,0.22,0.492127,219,ST-00199400
2,2026-05-20 22:38:00+00:00,0.06,0.134216,219,ST-00199400
3,2026-05-20 22:38:03+00:00,0.02,0.044739,219,ST-00199400
4,2026-05-20 22:38:06+00:00,0.01,0.022369,219,ST-00199400
5,2026-05-20 22:38:09+00:00,0.00,0.000000,0,ST-00199400
6,2026-05-20 22:38:15+00:00,0.00,0.000000,0,ST-00199400
7,2026-05-20 22:38:18+00:00,0.00,0.000000,0,ST-00199400
8,2026-05-20 22:38:21+00:00,0.00,0.000000,0,ST-00199400
9,2026-05-20 22:38:24+00:00,0.00,0.000000,0,ST-00199400


## 7. Hub & Device Status

In [9]:
def parse_hub_status(msg: dict) -> Dict:
    """Extract key fields from a hub_status message."""
    return {
        "serial_number": msg.get("serial_number"),
        "firmware_revision": msg.get("firmware_revision"),
        "uptime_s": msg.get("uptime"),
        "rssi_db": msg.get("rssi"),
        "timestamp": datetime.fromtimestamp(msg.get("timestamp", 0), tz=timezone.utc),
    }


def parse_device_status(msg: dict) -> Dict:
    """Extract key fields from a device_status message."""
    sensor_bits = msg.get("sensor_status", 0)
    return {
        "serial_number": msg.get("serial_number"),
        "firmware_revision": msg.get("firmware_revision"),
        "uptime_s": msg.get("uptime"),
        "battery_volts": msg.get("voltage"),
        "rssi_db": msg.get("rssi"),
        "sensor_status": f"0x{sensor_bits:04x}" if sensor_bits else "0x0000",
        "timestamp": datetime.fromtimestamp(msg.get("timestamp", 0), tz=timezone.utc),
    }


hub_msgs = [m for m in captured if m.get("type") == "hub_status"]
dev_msgs = [m for m in captured if m.get("type") == "device_status"]

if hub_msgs:
    print("=== Hub Status (latest) ===")
    display(pd.DataFrame([parse_hub_status(hub_msgs[-1])]).T.rename(columns={0: "value"}))
else:
    print("No hub_status messages in capture.")

if dev_msgs:
    print("\n=== Device Status (latest) ===")
    display(pd.DataFrame([parse_device_status(dev_msgs[-1])]).T.rename(columns={0: "value"}))
else:
    print("No device_status messages in capture.")

=== Hub Status (latest) ===


,value
serial_number,HB-00201189
firmware_revision,332
uptime_s,1002301
rssi_db,-38
timestamp,2026-05-20 22:38:06+00:00



=== Device Status (latest) ===


,value
serial_number,ST-00199400
firmware_revision,185
uptime_s,1057823
battery_volts,2.728
rssi_db,-56
sensor_status,0x0000
timestamp,2026-05-20 22:38:20+00:00


## 8. Live Stream

Prints a one-line summary for each message as it arrives.  
**Stop with Kernel → Interrupt** (or press the stop button in the toolbar).

In [11]:
def _summarise(msg: dict) -> str:
    """Return a short human-readable summary line for any message type."""
    msg_type = msg.get("type", "unknown")
    sn = msg.get("serial_number", "?")

    if msg_type == "hub_status":
        return f"hub_status    sn={sn}  uptime={msg.get('uptime')}s  rssi={msg.get('rssi')}dB"

    if msg_type == "device_status":
        return (
            f"device_status sn={sn}  battery={msg.get('voltage')}V  "
            f"rssi={msg.get('rssi')}dB"
        )

    if msg_type == "obs_st":
        obs = msg.get("obs", [[]])[0]
        temp = obs[7] if len(obs) > 7 else "?"
        wind = obs[2] if len(obs) > 2 else "?"
        return f"obs_st        sn={sn}  temp={temp}°C  wind_avg={wind}m/s"

    if msg_type == "rapid_wind":
        ob = msg.get("ob", [])
        speed_m_s = ob[1] if len(ob) > 1 else "?"
        speed_mph = ob[1] * 2.23694 if len(ob) > 1 else "?"
        direction = ob[2] if len(ob) > 2 else "?"
        return f"rapid_wind    sn={sn}  speed={round(speed_m_s, 2)}m/s  speed={round(speed_mph,2 )}mph dir={direction}°"

    if msg_type == "evt_precip":
        return f"evt_precip    sn={sn}  RAIN STARTED"

    if msg_type == "evt_strike":
        evt = msg.get("evt", [])
        dist = evt[1] if len(evt) > 1 else "?"
        energy = evt[2] if len(evt) > 2 else "?"
        return f"evt_strike    sn={sn}  dist={dist}km  energy={energy}"

    return f"{msg_type:20s} sn={sn}"


def stream_messages(max_messages: int = 200) -> None:
    """Print a summary line per message until max_messages or KeyboardInterrupt."""
    # sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    # sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    # sock.settimeout(1.0)
    # sock.bind(("", UDP_PORT))
    sock = get_sock()
    count = 0
    print("Streaming — interrupt the kernel to stop.\n")
    try:
        while count < max_messages:
            try:
                data, addr = sock.recvfrom(BUFFER_SIZE)
                msg = json.loads(data.decode("utf-8"))
                ts = datetime.now().strftime("%H:%M:%S")
                print(f"[{ts}] {_summarise(msg)}")
                count += 1
            except socket.timeout:
                pass
    except KeyboardInterrupt:
        print(f"\nStopped after {count} messages.")
    finally:
        sock.close()


stream_messages()

Streaming — interrupt the kernel to stop.

[18:41:57] rapid_wind    sn=ST-00199400  speed=0.0m/s  speed=0.0mph dir=0°
[18:42:00] rapid_wind    sn=ST-00199400  speed=0.0m/s  speed=0.0mph dir=0°
[18:42:03] rapid_wind    sn=ST-00199400  speed=0.0m/s  speed=0.0mph dir=0°
[18:42:06] hub_status    sn=HB-00201189  uptime=1002541s  rssi=-38dB
[18:42:06] rapid_wind    sn=ST-00199400  speed=0.0m/s  speed=0.0mph dir=0°
[18:42:09] rapid_wind    sn=ST-00199400  speed=0.0m/s  speed=0.0mph dir=0°
[18:42:12] rapid_wind    sn=ST-00199400  speed=0.0m/s  speed=0.0mph dir=0°
[18:42:15] rapid_wind    sn=ST-00199400  speed=0.0m/s  speed=0.0mph dir=0°
[18:42:19] rapid_wind    sn=ST-00199400  speed=0.26m/s  speed=0.58mph dir=87°
[18:42:20] device_status sn=ST-00199400  battery=2.722V  rssi=-57dB
[18:42:20] obs_st        sn=ST-00199400  temp=28.87°C  wind_avg=0.08m/s
[18:42:21] rapid_wind    sn=ST-00199400  speed=0.07m/s  speed=0.16mph dir=87°
[18:42:24] rapid_wind    sn=ST-00199400  speed=0.02m/s  speed=0.04m